# EESD Hindi — Training + Evaluation Notebook
**Team Transformers** · April 2026

**Full pipeline:** trains exit heads on AI4Bharat Hindi, then evaluates 7 speculative decoding methods on 100 XL-Sum Hindi prompts.

**Estimated runtime:**
- Training: ~6–8 hours on A100 (3 epochs, 167K samples) · **~80–100 CU**
- Evaluation: ~3–4 hours on A100 · **~45–60 CU**

---

### 📁 Required Google Drive structure

Everything lives under **`MyDrive/EESD_project/`** (your parent folder).
Upload **only the source code** before running — no checkpoint needed (we train it here):

```
MyDrive/EESD_project/          ← DRIVE_ROOT (parent directory)
├── src/
│   ├── __init__.py
│   ├── model.py
│   ├── inference.py
│   ├── evaluate_all.py
│   ├── draft_model_baseline.py
│   ├── data.py               ← training data loader
│   └── train.py              ← training script
└── EESD/                     ← created automatically during training
    ├── exit_heads_final.pt   ← saved here after training
    └── loss_history.json     ← saved after each epoch
```

Results are saved back to:
```
MyDrive/EESD_project/outputs/  ← created automatically
    ├── ablation_results.json
    ├── ablation_results_partial.json
    └── ablation_results.log
```

---

### 📊 How data is accessed

**Training data (AI4Bharat IndicCorpv2 Hindi):** downloaded automatically via HuggingFace Hub at runtime.
~27 GB raw text (only 167 K lines are read). Requires ~30–50 GB free disk space on Colab (`/content`).

**Evaluation data (XL-Sum Hindi):** fetched at runtime via a HuggingFace parquet API call — no upload needed.

---

### ✅ Pre-flight checklist
1. **Runtime → Change runtime type → A100 GPU**
2. Upload `src/` files (all 7) to `MyDrive/EESD_project/src/` in Google Drive
3. Run cells **top to bottom, one at a time**
4. If the session disconnects during training, rerun from **Step 5b (Resume training)** — checkpoints are auto-saved to Drive after each epoch

## Step 1 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


## Step 2 — Copy project files from Drive to Colab

In [4]:
import os, shutil, time
from google.colab import drive as _colab_drive

DRIVE_ROOT = '/content/drive/MyDrive/EESD_project'
WORK_DIR   = '/content/eesd'

# Verify Drive folder exists
assert os.path.isdir(DRIVE_ROOT), (
    f'Could not find {DRIVE_ROOT}. '
    'Create the folder in Google Drive and upload files as described above.'
)

os.makedirs(f'{WORK_DIR}/src',     exist_ok=True)
os.makedirs(f'{WORK_DIR}/EESD',    exist_ok=True)
os.makedirs(f'{WORK_DIR}/outputs', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results/figures',     exist_ok=True)

# ---------------------------------------------------------------------------
# Robust chunked copy — avoids Drive FUSE 'Transport endpoint not connected'
# error that occurs with shutil.copy2 on files > ~1 GB
# ---------------------------------------------------------------------------
def copy_large_file(src, dst, chunk_mb=64, max_retries=5):
    chunk = chunk_mb * 1024 * 1024
    total = os.path.getsize(src)
    for attempt in range(1, max_retries + 1):
        try:
            written = 0
            with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
                while True:
                    buf = fsrc.read(chunk)
                    if not buf:
                        break
                    fdst.write(buf)
                    written += len(buf)
                    pct = written / total * 100
                    print(f'\r  {pct:.1f}%  ({written/1e9:.2f} / {total/1e9:.2f} GB)',
                          end='', flush=True)
            print(f'\r  100.0%  ({total/1e9:.2f} GB) — done          ')
            return
        except OSError as e:
            print(f'\n  OSError on attempt {attempt}/{max_retries}: {e}')
            if attempt < max_retries:
                print('  Remounting Drive and retrying in 10 s...')
                time.sleep(10)
                _colab_drive.mount('/content/drive', force_remount=True)
            else:
                raise RuntimeError(
                    f'Failed to copy {src} after {max_retries} attempts.\n'
                    'Alternative: upload the file directly via the '
                    'Colab file panel (left sidebar → upload).'
                ) from e

# --- Copy Python source files ---
src_files = [
    '__init__.py', 'model.py', 'inference.py',
    'evaluate_all.py', 'draft_model_baseline.py',
    'data.py', 'train.py',
]
for f in src_files:
    src  = f'{DRIVE_ROOT}/src/{f}'
    dest = f'{WORK_DIR}/src/{f}'
    assert os.path.exists(src), f'Missing: {src}'
    shutil.copy2(src, dest)
    print(f'  Copied src/{f}')

# --- Copy existing checkpoint if present (optional — for resume or eval-only run) ---
ckpt_src = f'{DRIVE_ROOT}/EESD/exit_heads_final.pt'
ckpt_dst = f'{WORK_DIR}/EESD/exit_heads_final.pt'
if os.path.exists(ckpt_src):
    print('\nFound existing checkpoint — copying (chunked)...')
    copy_large_file(ckpt_src, ckpt_dst)
    print(f'  Verified: {os.path.getsize(ckpt_dst)/1e9:.2f} GB written to {ckpt_dst}')
else:
    print('\nNo existing checkpoint found — will train from scratch.')

# --- Copy bottleneck checkpoint if it exists on Drive ---
bn_src = f'{DRIVE_ROOT}/EESD/bottleneck_exit_heads_final.pt'
bn_dst = f'{WORK_DIR}/EESD/bottleneck_exit_heads_final.pt'
if os.path.exists(bn_src):
    shutil.copy2(bn_src, bn_dst)
    print(f'Copied bottleneck checkpoint: {os.path.getsize(bn_dst)/1e6:.1f} MB')
else:
    print('No bottleneck checkpoint on Drive — method 5 will be skipped')

# --- Copy per-epoch checkpoints for resume (if any) ---
drive_ckpt_dir = f'{DRIVE_ROOT}/EESD/checkpoints'
if os.path.isdir(drive_ckpt_dir):
    os.makedirs(f'{WORK_DIR}/results/checkpoints', exist_ok=True)
    for fname in os.listdir(drive_ckpt_dir):
        if fname.endswith('.pt') or fname.endswith('.json'):
            shutil.copy2(f'{drive_ckpt_dir}/{fname}',
                         f'{WORK_DIR}/results/checkpoints/{fname}')
            print(f'  Copied checkpoints/{fname}')

# --- Copy loss history if present ---
loss_src = f'{DRIVE_ROOT}/EESD/loss_history.json'
if os.path.exists(loss_src):
    shutil.copy2(loss_src, f'{WORK_DIR}/results/checkpoints/loss_history.json')
    print('  Copied loss_history.json')

os.chdir(WORK_DIR)
print(f'\nWorking directory: {os.getcwd()}')

  Copied src/__init__.py
  Copied src/model.py
  Copied src/inference.py
  Copied src/evaluate_all.py
  Copied src/draft_model_baseline.py
  Copied src/data.py
  Copied src/train.py

Found existing checkpoint — copying (chunked)...
  100.0%  (2.80 GB) — done          
  Verified: 2.80 GB written to /content/eesd/EESD/exit_heads_final.pt
  Copied checkpoints/exit_heads_epoch1.pt
  Copied checkpoints/exit_heads_epoch2.pt


KeyboardInterrupt: 

## Step 3 — Install dependencies

In [ ]:
# Install all required packages
!pip install -q \
    torch torchvision \
    transformers==4.44.2 \
    accelerate \
    bitsandbytes \
    datasets \
    pandas \
    pyarrow \
    tqdm \
    numpy \
    sentencepiece \
    huggingface_hub \
    matplotlib

print('Dependencies installed.')

## Step 4 — Verify setup

In [ ]:
import torch, os

print('=== GPU ===')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  {gpu}  ({vram_gb:.1f} GB VRAM)')
    if vram_gb < 30:
        print('  ⚠️  WARNING: Less than 30 GB VRAM. Training may OOM — try --batch_size 1.')
else:
    print('  ❌ No GPU found! Change runtime type to A100.')

print('\n=== Source files ===')
src_required = [
    'src/__init__.py', 'src/model.py', 'src/inference.py',
    'src/data.py', 'src/train.py',
    'src/evaluate_all.py', 'src/draft_model_baseline.py',
]
all_ok = True
for f in src_required:
    exists = os.path.exists(f)
    size   = f'({os.path.getsize(f)/1024:.0f} KB)' if exists else ''
    status = '✅' if exists else '❌  MISSING'
    print(f'  {status}  {f}  {size}')
    if not exists:
        all_ok = False

print('\n=== Checkpoint (optional — trained in Step 6) ===')
ckpt = 'EESD/exit_heads_final.pt'
if os.path.exists(ckpt):
    print(f'  ✅  {ckpt}  ({os.path.getsize(ckpt)/1e9:.2f} GB)  — will SKIP training, go straight to eval')
else:
    print(f'  ℹ️   {ckpt} not found — training required (Step 6)')

print('\n=== Disk space ===')
!df -h /content | tail -1
print('  ⚠️  Training downloads ~27 GB of AI4Bharat text. Ensure ≥ 50 GB free.')

if all_ok:
    print('\n✅ Source files OK — proceed to training (Step 5).')
else:
    print('\n❌ Fix missing source files above before continuing.')

---
## PART A — Training Exit Heads

Trains three exit heads (at layers 8, 16, 22) on 167K Hindi sentences using **KL-divergence distillation** from the frozen Qwen2-1.5B full model.

- **Loss:** soft KL divergence (temperature T=2.0) from full-model logits to each exit head
- **Only the exit heads are trained** — the base model stays frozen
- **Saves:** per-epoch checkpoints + loss history to Drive after each epoch
- **Resume:** if the session disconnects, run Step 5b and then Step 6 again

> **Expected training time:** ~2–2.5 h/epoch × 3 epochs = **~6–8 h total** on A100

## Step 5a — Configure training

In [ ]:
import os, json

# ── Training hyperparameters ─────────────────────────────────────────────────
MODEL_NAME      = 'Qwen/Qwen2-1.5B'
EXIT_DEPTHS     = [8, 16, 22]          # exit head positions
EPOCHS          = 10
BATCH_SIZE      = 2
GRAD_ACCUM      = 4                    # effective batch = 8
LR              = 1e-4
TEMPERATURE     = 2.0
MAX_LENGTH      = 256
MAX_SAMPLES     = 167_000             # ~5M tokens; reduce to 50000 if low on disk
CACHE_DIR       = '/content/cache'    # HF download cache (separate from /content/eesd)
OUTPUT_DIR      = '/content/drive/MyDrive/EESD_project/results/checkpoints'
DRIVE_CKPT_DIR  = f'{DRIVE_ROOT}/EESD/checkpoints'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Check if we should resume from a previous epoch
state_path = f'{OUTPUT_DIR}/training_state.json'
RESUME_FROM  = None
START_EPOCH  = 4

if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    completed = state.get('completed_epoch', 0)
    if completed >= EPOCHS:
        print(f'Training already complete ({completed}/{EPOCHS} epochs). Skipping to evaluation.')
    elif completed > 0:
        RESUME_FROM = state.get('checkpoint', f'{OUTPUT_DIR}/exit_heads_epoch{completed}.pt')
        START_EPOCH = completed + 1
        print(f'Will resume from epoch {START_EPOCH} using: {RESUME_FROM}')
    else:
        print(f'Training state found but completed_epoch=0 — starting from scratch.')
else:
    print(f'No training state found — starting from epoch 1.')

depths_str = ' '.join(str(d) for d in EXIT_DEPTHS)

# Build the training command
TRAIN_CMD = (
    f'python -u -m src.train'
    f' --model_name {MODEL_NAME}'
    f' --exit_depths {depths_str}'
    f' --epochs {EPOCHS}'
    f' --batch_size {BATCH_SIZE}'
    f' --gradient_accumulation_steps {GRAD_ACCUM}'
    f' --lr {LR}'
    f' --temperature {TEMPERATURE}'
    f' --max_length {MAX_LENGTH}'
    f' --max_samples {MAX_SAMPLES}'
    f' --num_workers 2'
    f' --output_dir {OUTPUT_DIR}'
    f' --cache_dir {CACHE_DIR}'
    f' --platform colab'
)
if RESUME_FROM:
    TRAIN_CMD += f' --resume_from {RESUME_FROM} --start_epoch {START_EPOCH}'

print('\n=== Training command ===')
print(TRAIN_CMD)
print(f'\nEffective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Starting epoch: {START_EPOCH}/{EPOCHS}')

## Step 5b — Restore checkpoints from Drive (run only after a session disconnect)

In [ ]:
# Run this cell if the session disconnected during training.
# It restores all per-epoch checkpoints and training state from Drive.
import os, shutil, json

DRIVE_ROOT     = '/content/drive/MyDrive/EESD_project'
DRIVE_CKPT_DIR = f'{DRIVE_ROOT}/EESD/checkpoints'
OUTPUT_DIR     = '/content/drive/MyDrive/EESD_project/results/checkpoints'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(DRIVE_CKPT_DIR):
    print(f'No checkpoint directory found at {DRIVE_CKPT_DIR}')
    print('Nothing to restore — training will start from epoch 1.')
else:
    restored = []
    for fname in sorted(os.listdir(DRIVE_CKPT_DIR)):
        if fname.endswith('.pt') or fname.endswith('.json'):
            src = f'{DRIVE_CKPT_DIR}/{fname}'
            dst = f'{OUTPUT_DIR}/{fname}'
            shutil.copy2(src, dst)
            restored.append(fname)
            print(f'  Restored: {fname}  ({os.path.getsize(dst)/1e6:.0f} MB)')

    if not restored:
        print('No checkpoint files found in Drive — starting from scratch.')
    else:
        state_path = f'{OUTPUT_DIR}/training_state.json'
        if os.path.exists(state_path):
            with open(state_path) as f:
                state = json.load(f)
            print(f"\nLast completed epoch: {state.get('completed_epoch', '?')}")
            print(f"Checkpoint: {state.get('checkpoint', '?')}")
            print(f"Train loss: {state.get('train_loss', '?')}")
        print('\n✅ Re-run Step 5a to update RESUME_FROM, then run Step 6 to continue training.')

## Step 6 — Run training

> **This takes ~6–8 hours on A100.** Checkpoints are saved after every epoch.
> The cell streams live output. If it disconnects, run Step 5b then re-run Step 5a and this cell.

In [ ]:
import subprocess, sys, os, shutil, json, time, shlex
from google.colab import drive as _colab_drive

DRIVE_ROOT     = '/content/drive/MyDrive/EESD_project'
DRIVE_CKPT_DIR = f'{DRIVE_ROOT}/EESD/checkpoints'
OUTPUT_DIR     = '/content/drive/MyDrive/EESD_project/results/checkpoints'

# Check if training is already done
state_path = f'{OUTPUT_DIR}/training_state.json'
if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    if state.get('completed_epoch', 0) >= EPOCHS:
        print(f'Training already complete ({EPOCHS} epochs). Skipping to Step 7.')
        raise SystemExit(0)  # stop cell without error

print('=== Starting training ===')
print(TRAIN_CMD)
print('─' * 60)

def _sync_checkpoints_to_drive():
    """Copy all .pt and .json files from OUTPUT_DIR to DRIVE_CKPT_DIR."""
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
    synced = []
    for fname in os.listdir(OUTPUT_DIR):
        if fname.endswith('.pt') or fname.endswith('.json'):
            src = f'{OUTPUT_DIR}/{fname}'
            dst = f'{DRIVE_CKPT_DIR}/{fname}'
            try:
                shutil.copy2(src, dst)
                synced.append(fname)
            except Exception as e:
                print(f'  Drive sync warning: {e}')
    if synced:
        print(f'  [Drive] Synced {len(synced)} files to {DRIVE_CKPT_DIR}')

# shlex.split handles backslash-newline continuations correctly;
# plain .split() would pass each backslash as a stray argument to argparse
proc = subprocess.Popen(
    shlex.split(TRAIN_CMD),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)
    if 'Checkpoint saved' in line or 'Training state saved' in line:
        _sync_checkpoints_to_drive()

proc.wait()

if proc.returncode == 0:
    print('\n✅ Training complete.')
    _sync_checkpoints_to_drive()
else:
    print(f'\n❌ Training exited with code {proc.returncode}. Check output above.')
    print('Syncing whatever was completed to Drive...')
    _sync_checkpoints_to_drive()


## Step 7 — Save trained checkpoint to Drive

In [ ]:
import os, shutil, json

DRIVE_ROOT = '/content/drive/MyDrive/EESD_project'
OUTPUT_DIR = '/content/drive/MyDrive/EESD_project/results/checkpoints'

os.makedirs(f'{DRIVE_ROOT}/EESD', exist_ok=True)

# Copy exit_heads_final.pt to Drive + to the EESD/ folder that eval expects
final_local = f'{OUTPUT_DIR}/exit_heads_final.pt'
final_eesd  = 'EESD/exit_heads_final.pt'
final_drive = f'{DRIVE_ROOT}/EESD/exit_heads_final.pt'

if not os.path.exists(final_local):
    print(f'❌ {final_local} not found — training may not have completed.')
    print('   Check training_state.json:', end=' ')
    if os.path.exists(f'{OUTPUT_DIR}/training_state.json'):
        with open(f'{OUTPUT_DIR}/training_state.json') as f:
            print(json.load(f))
    else:
        print('not found either')
else:
    size_gb = os.path.getsize(final_local) / 1e9
    print(f'Found {final_local}  ({size_gb:.2f} GB)')

    # Copy to EESD/ for evaluation
    shutil.copy2(final_local, final_eesd)
    print(f'  Copied → {final_eesd}')

    # Copy to Drive
    print(f'  Copying to Drive ({size_gb:.2f} GB)...')
    shutil.copy2(final_local, final_drive)
    print(f'  Copied → {final_drive}')

# Copy loss_history.json to Drive
loss_local = f'{OUTPUT_DIR}/loss_history.json'
if os.path.exists(loss_local):
    shutil.copy2(loss_local, f'{DRIVE_ROOT}/EESD/loss_history.json')
    shutil.copy2(loss_local, f'{WORK_DIR}/results/checkpoints/loss_history.json')
    print(f'  Copied loss_history.json to Drive')

# Copy figures
curve_local = 'results/figures/loss_curve.png'
if os.path.exists(curve_local):
    shutil.copy2(curve_local, f'{DRIVE_ROOT}/EESD/loss_curve.png')
    print(f'  Copied loss_curve.png to Drive')

print('\n✅ Checkpoint saved. Proceed to evaluation (Step 8).')

---
## PART B — Evaluation

Runs all 7 speculative decoding methods on 100 XL-Sum Hindi prompts:
`autoregressive`, `draft_model`, `eesd_heavy_hook`, `eesd_heavy_true_exit`,
`eesd_bottleneck_true_exit`, `eesd_thompson`, `eesd_entropy_exit`.

Results are checkpointed after each method and auto-synced to Drive.

## Step 8 — Verify checkpoint before evaluation

In [ ]:
import torch, os

print('=== GPU ===')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  {gpu}  ({vram_gb:.1f} GB VRAM)')
    if vram_gb < 30:
        print('  ⚠️  WARNING: Less than 30 GB VRAM. Consider --eval_samples 50 if OOM.')
else:
    print('  ❌ No GPU found! Change runtime type to A100.')

print('\n=== Files ===')
required = [
    'src/__init__.py', 'src/model.py', 'src/inference.py',
    'src/evaluate_all.py', 'src/draft_model_baseline.py',
    'EESD/exit_heads_final.pt',
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = f'({os.path.getsize(f)/1e6:.0f} MB)' if exists else ''
    status = '✅' if exists else '❌  MISSING'
    print(f'  {status}  {f}  {size}')
    if not exists:
        all_ok = False

print('\n=== Disk space ===')
!df -h /content | tail -1

if all_ok:
    print('\n✅ All checks passed — ready for evaluation.')
else:
    print('\n❌ Fix missing files. If checkpoint is missing, complete Steps 5–7 first.')

## Step 9 — Run evaluation

Runs all 7 methods on 100 Hindi prompts.
Progress is printed live and checkpointed to `outputs/ablation_results_partial.json` after each method.

> **If this cell times out or Colab disconnects**, run the "Recover partial results" cell below — your completed steps are already saved.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'src.evaluate_all',
    '--eval_samples',   '100',
    '--max_new_tokens', '50',
    '--K',              '3',
    '--output_path',    '/content/drive/MyDrive/EESD_project/outputs/ablation_results.json',
]

print('Running:', ' '.join(cmd))
print('Logs → outputs/ablation_results.log')
print('Checkpoint → outputs/ablation_results_partial.json')
print('─' * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n✅ Evaluation complete.')
else:
    print(f'\n❌ Process exited with code {proc.returncode}. Check the output above.')

## Step 10 — Recover partial results (run if Step 9 crashed)
If the evaluation crashed mid-way, the partial results are still saved.

In [ ]:
import json, os

partial_path = '/content/drive/MyDrive/EESD_project/outputs/ablation_results_partial.json'
final_path   = '/content/drive/MyDrive/EESD_project/outputs/ablation_results.json'

path = final_path if os.path.exists(final_path) else partial_path

if not os.path.exists(path):
    print('No results file found yet — evaluation may not have started.')
else:
    with open(path) as f:
        results = json.load(f)
    completed = [k for k in results if k not in
                 ('k_ablation', 'prompt_length_ablation', 'losslessness',
                  'morphological_analysis', 'latency_breakdown', 'cross_lingual')]
    print(f'Loaded from: {path}')
    print(f'Completed methods: {completed}\n')
    for method in completed:
        r = results[method]
        if r.get('skipped'):
            print(f'  {method:<35} SKIPPED')
        else:
            alpha   = r.get('alpha', 'n/a')
            speedup = r.get('speedup', 'n/a')
            print(f'  {method:<35} α={alpha}   speedup={speedup}x')

## Step 11 — Results summary table

In [ ]:
import json, pandas as pd

with open('/content/drive/MyDrive/EESD_project/outputs/ablation_results.json') as f:
    results = json.load(f)

METHOD_KEYS = [
    'autoregressive', 'draft_model', 'eesd_heavy_hook',
    'eesd_heavy_true_exit', 'eesd_bottleneck_true_exit',
    'eesd_thompson', 'eesd_entropy_exit',
]
METHOD_LABELS = [
    'Autoregressive', 'Draft Model (0.5B)', 'EESD Hook',
    'EESD True Exit', 'EESD Bottleneck',
    'Thompson Sampling', 'Entropy Exit',
]

rows = []
for key, label in zip(METHOD_KEYS, METHOD_LABELS):
    r = results.get(key, {})
    if r.get('skipped'):
        rows.append({'Method': label, 'α': '—', 'Speedup': '—',
                     'tok/s': '—', 'VRAM (MB)': '—', 'Note': 'SKIPPED'})
    else:
        rows.append({
            'Method':    label,
            'α':         r.get('alpha',         '—'),
            'Speedup':   str(r.get('speedup',   '—')) + 'x',
            'tok/s':     r.get('tokens_per_sec','—'),
            'VRAM (MB)': r.get('vram_mb',       '—'),
            'Note': '',
        })

df = pd.DataFrame(rows)
print('=== 7-Method Comparison ===')
display(df)

# K-ablation
if 'k_ablation' in results:
    print('\n=== K-Ablation (depth=22) ===')
    k_rows = [{'K': k, 'α': v['alpha'], 'Speedup': str(v['speedup'])+'x'}
              for k, v in results['k_ablation'].items()]
    display(pd.DataFrame(k_rows))

# Morphological analysis
if 'morphological_analysis' in results:
    print('\n=== Morphological Analysis ===')
    m = results['morphological_analysis']
    morph_rows = [{'Category': cat, 'α': v['alpha'], 'Count': v['count']}
                  for cat, v in m.items()]
    display(pd.DataFrame(morph_rows))

# Cross-lingual
if 'cross_lingual' in results:
    cl = results['cross_lingual']
    print(f'\n=== Cross-Lingual (depth=22, K=3) ===')
    print(f'  Hindi α  = {cl["hindi_alpha"]}  (n={cl["hindi_samples"]})')
    print(f'  English α = {cl["english_alpha"]}  (n={cl["english_samples"]})')

# Losslessness
if 'losslessness' in results:
    ll = results['losslessness']
    print(f'\n=== Losslessness ===')
    print(f'  Exact match: {ll["exact_match"]}/{ll["total_prompts"]}  ({ll["match_rate"]*100:.1f}%)')

## Step 12 — View training loss curves

In [ ]:
import json, os, matplotlib.pyplot as plt

# Look in both locations
loss_candidates = [
    '/content/drive/MyDrive/EESD_project/results/checkpoints/loss_history.json',
    '/content/drive/MyDrive/EESD_project/EESD/loss_history.json',
]
loss_path = next((p for p in loss_candidates if os.path.exists(p)), None)

if loss_path is None:
    print('loss_history.json not found — skipping.')
else:
    with open(loss_path) as f:
        history = json.load(f)

    epochs     = [e['epoch']      for e in history]
    train_loss = [e['train_loss'] for e in history]
    val_loss   = [e['val_loss']   for e in history]

    DEPTHS = [8, 16, 22]
    colors = ['#e15759', '#4e79a7', '#59a14f']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Overall loss
    axes[0].plot(epochs, train_loss, 'o-', label='Train')
    axes[0].plot(epochs, val_loss,   's--', label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('KL Loss')
    axes[0].set_title('Overall Training Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Per-depth losses
    has_depth = any(f'train_depth_losses' in e for e in history)
    if has_depth:
        for depth, color in zip(DEPTHS, colors):
            dk = str(depth)
            t_vals = [e['train_depth_losses'].get(dk, None) for e in history]
            v_vals = [e['val_depth_losses'].get(dk, None)   for e in history]
            if any(v is not None for v in t_vals):
                axes[1].plot(epochs, t_vals, 'o-',  color=color, label=f'Layer {depth} train')
                axes[1].plot(epochs, v_vals, 's--', color=color, label=f'Layer {depth} val', alpha=0.75)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('KL Loss')
        axes[1].set_title('Loss per Exit Depth')
        axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'Per-depth losses not available',
                     ha='center', va='center', transform=axes[1].transAxes)

    # Per-depth acceptance rate
    if any('acceptance_rate' in e for e in history):
        fig2, ax2 = plt.subplots(figsize=(6, 3))
        for depth, color in zip(DEPTHS, colors):
            dk = str(depth)
            alphas = [e['acceptance_rate'].get(dk, None) for e in history
                      if 'acceptance_rate' in e]
            ep_with_alpha = [e['epoch'] for e in history if 'acceptance_rate' in e]
            ax2.plot(ep_with_alpha, alphas, 'o-', color=color, label=f'Layer {depth}')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acceptance rate α')
        ax2.set_title('Acceptance Rate per Epoch (draft→verify)')
        ax2.legend(); ax2.grid(True, alpha=0.3)
        fig2.tight_layout()
        fig2.savefig('outputs/acceptance_rate_curve.png', dpi=150, bbox_inches='tight')
        plt.show()

    plt.tight_layout()
    os.makedirs('outputs', exist_ok=True)
    fig.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to outputs/training_curves.png')

## Step 13 — Save everything to Google Drive

In [ ]:
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/EESD_project'
DRIVE_OUT  = f'{DRIVE_ROOT}/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)

files_to_save = [
    'outputs/ablation_results.json',
    'outputs/ablation_results_partial.json',
    'outputs/ablation_results.log',
    'outputs/training_curves.png',
    'outputs/acceptance_rate_curve.png',
]

for p in files_to_save:
    if os.path.exists(p):
        dest = os.path.join(DRIVE_OUT, os.path.basename(p))
        shutil.copy2(p, dest)
        size = os.path.getsize(p) / 1024
        print(f'  ✅  {p}  ({size:.1f} KB)  →  {dest}')
    else:
        print(f'  ⚠️   {p} not found — skipped')

print(f'\nAll files saved to {DRIVE_OUT}')

## Step 14 — Download results to your local machine

In [ ]:
from google.colab import files
import os

to_download = [
    'outputs/ablation_results.json',
    'outputs/ablation_results.log',
    'outputs/training_curves.png',
    'outputs/acceptance_rate_curve.png',
]

for p in to_download:
    if os.path.exists(p):
        print(f'Downloading {p}...')
        files.download(p)
    else:
        print(f'  Skipped (not found): {p}')

---
## Appendix — OOM / crash recovery

**If training OOMs:**
```python
# In Step 5a, set:
BATCH_SIZE  = 1
GRAD_ACCUM  = 8   # same effective batch size
```

**If training session disconnects:**
1. Remount Drive (Step 1)
2. Copy files (Step 2) — restores checkpoints from `EESD/checkpoints/`
3. Install deps (Step 3)
4. Run Step 5b to restore checkpoint files
5. Run Step 5a — it will auto-detect the last epoch and set `RESUME_FROM`
6. Run Step 6 to continue from where it stopped

**If evaluation crashes:**
- Partial results are in `outputs/ablation_results_partial.json` and synced to Drive
- Run Step 10 to inspect completed methods

**To check GPU utilisation while running:**
```python
!nvidia-smi
```

**To check disk space:**
```python
!df -h /content
```

**To check training log tail:**
```python
!ls -lh results/checkpoints/
```